# Penyelesaian Persamaan Non Linear - Metode Bisection (Bagi Dua)

**Nama:** [ISI NAMA ANDA] | **NIM:** [ISI NIM] | **Mata Kuliah:** Komputasi Numerik

Fungsi yang dicari akarnya: **f(x) = x³ - x - 2**, pada selang awal [1, 2].
Jalankan tiap cell berurutan dengan **Shift + Enter**.

## Cell 1 - Import library
NumPy untuk hitung, pandas untuk tabel iterasi, matplotlib untuk grafik.

In [ ]:
import numpy as np  # operasi numerik
import pandas as pd  # membuat tabel iterasi
import matplotlib.pyplot as plt  # menggambar grafik

## Cell 2 - Definisi fungsi
Ubah isi fungsi untuk mencoba persamaan lain.

In [ ]:
def f(x):  # fungsi yang akan dicari akarnya
    return x**3 - x - 2  # f(x) = x^3 - x - 2

## Cell 3 - Mengecek selang awal
Bisection hanya valid bila f(a) dan f(b) berbeda tanda (f(a)·f(b) < 0), sesuai Teorema Nilai Antara.

In [ ]:
a, b = 1.0, 2.0  # batas kiri dan kanan selang awal
print("f(a) =", f(a))  # nilai fungsi di batas kiri (negatif)
print("f(b) =", f(b))  # nilai fungsi di batas kanan (positif)
print("Ada akar di [a,b]?", f(a) * f(b) < 0)  # True berarti tanda berbeda sehingga akar terjamin ada

## Cell 4 - Fungsi Bisection
Setiap iterasi: titik tengah c = (a+b)/2, lalu pilih setengah selang yang tandanya masih berbeda. Berhenti jika setengah lebar selang < toleransi.

In [ ]:
def bisection(f, a, b, tol=1e-6, maks_iter=100):  # input: fungsi, selang, toleransi, batas iterasi
    fa = f(a)  # nilai fungsi di a, disimpan agar tidak dihitung ulang
    if fa * f(b) > 0:  # jika tanda sama, metode tidak bisa dipakai
        raise ValueError("f(a) dan f(b) harus berbeda tanda")  # hentikan dengan pesan
    riwayat = []  # list untuk mencatat tiap iterasi
    for i in range(1, maks_iter + 1):  # ulangi sampai maksimum iterasi
        c = (a + b) / 2  # titik tengah selang
        fc = f(c)  # nilai fungsi di titik tengah
        galat = (b - a) / 2  # perkiraan galat = setengah lebar selang
        riwayat.append([i, a, b, c, fc, galat])  # catat baris tabel iterasi
        if fc == 0 or galat < tol:  # akar tepat ditemukan atau galat sudah cukup kecil
            break  # keluar dari perulangan
        if fa * fc < 0:  # akar berada di setengah kiri [a, c]
            b = c  # geser batas kanan ke c
        else:  # akar berada di setengah kanan [c, b]
            a = c  # geser batas kiri ke c
            fa = fc  # perbarui f(a) mengikuti a baru
    tabel = pd.DataFrame(riwayat, columns=["iterasi", "a", "b", "c", "f(c)", "galat"])  # ubah catatan jadi tabel
    return c, tabel  # kembalikan akar hampiran dan tabelnya

## Cell 5 - Menjalankan dan menampilkan tabel

In [ ]:
akar, tabel = bisection(f, 1.0, 2.0, tol=1e-6)  # jalankan dengan toleransi 0,000001
pd.set_option("display.float_format", lambda v: f"{v:.7f}")  # tampilkan 7 desimal
print(tabel.to_string(index=False))  # cetak tabel tanpa nomor indeks pandas
print()  # baris kosong
print(f"Akar hampiran : {akar:.7f}")  # hasil akhir
print(f"f(akar)       : {f(akar):.2e}")  # nilai fungsi di akar, mendekati nol
print(f"Jumlah iterasi: {len(tabel)}")  # banyaknya iterasi yang dipakai

## Cell 6 - Verifikasi
Banyak iterasi teoritis: n ≥ log2((b-a)/tol). Akar juga dibandingkan dengan solusi polinomial NumPy.

In [ ]:
n_teori = int(np.ceil(np.log2((2.0 - 1.0) / 1e-6)))  # batas iterasi dari rumus galat bisection
print("Iterasi teoritis :", n_teori)  # bandingkan dengan jumlah iterasi di tabel
akar_np = [r.real for r in np.roots([1, 0, -1, -2]) if abs(r.imag) < 1e-9][0]  # akar real dari x^3 - x - 2 = 0
print("Akar NumPy       :", round(akar_np, 7))  # pembanding
print("Selisih          :", abs(akar - akar_np))  # harus lebih kecil dari toleransi

## Cell 7 - Grafik
Kiri: kurva f(x) dan akar. Kanan: galat per iterasi (skala log), turun separuh tiap langkah.

In [ ]:
xs = np.linspace(0.8, 2.2, 200)  # titik-titik x untuk menggambar kurva
fig, ax = plt.subplots(1, 2, figsize=(11, 4))  # dua panel berdampingan
ax[0].plot(xs, f(xs), label="f(x)")  # kurva fungsi
ax[0].axhline(0, color="gray", linewidth=0.8)  # garis sumbu x
ax[0].scatter([akar], [0], color="red", zorder=3, label="akar")  # titik akar
ax[0].set_title("Kurva f(x) dan akar")  # judul panel kiri
ax[0].legend()  # tampilkan legenda
ax[1].semilogy(tabel["iterasi"], tabel["galat"], marker="o")  # galat vs iterasi, sumbu y logaritmik
ax[1].set_title("Penurunan galat tiap iterasi")  # judul panel kanan
ax[1].set_xlabel("iterasi")  # label sumbu x
plt.tight_layout()  # rapikan jarak
plt.show()  # tampilkan grafik